# Notebook 2: Documents & Chunking

**Difficulty:** Beginner-Intermediate | **Estimated Time:** 90-120 minutes

## Learning Objectives

By the end of this notebook, you will be able to:

1. ✅ Load documents from multiple sources (local files, PDFs, web)
2. ✅ Implement different chunking strategies (sentence, token, semantic)
3. ✅ Add custom metadata at document and node levels
4. ✅ Create and manage node relationships
5. ✅ Optimize chunking for retrieval quality
6. ✅ Apply batch embedding optimization

## Prerequisites

- Completed Notebook 1: Setup & Basics
- Understanding of embeddings and chunking concepts
- Sample PDFs in `data/research_papers/` directory

## Curriculum Coverage

- **Section 2.1:** Loading Documents from Various Sources
- **Section 2.2:** Document Preprocessing
- **Section 2.3:** Document Parsing and Chunking
- **Section 2.4:** Metadata Management
- **Section 2.5:** Document Nodes

---

## 1. Setup & Imports

In [2]:
# Core LlamaIndex
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, Document
from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo

# Node Parsers (Chunking Strategies)
from llama_index.core.node_parser import (
    SentenceSplitter,
    TokenTextSplitter,
    SemanticSplitterNodeParser,
)

# Metadata Extraction
from llama_index.core.extractors import (
    TitleExtractor,
    SummaryExtractor,
)
from llama_index.core.ingestion import IngestionPipeline

# LLM and Embeddings
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

# Utilities
from dotenv import load_dotenv
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful!")

✅ Imports successful!


In [3]:
# Load environment variables and configure Settings
load_dotenv()

# Configure LLM
Settings.llm = AzureOpenAI(
    engine=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    model="gpt-4.1-mini",
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-10-21",
    temperature=0,
)

# Configure Embedding Model
Settings.embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small",
    deployment_name=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-10-21",
)
print("✅ Settings configured")

✅ Settings configured


---

## 2. Loading Documents from Multiple Sources

### 2.1 Local File Loading with SimpleDirectoryReader

In [4]:
# Check data directory structure
data_dir = Path("./data")
sample_docs_dir = data_dir / "sample_docs"
research_papers_dir = data_dir / "research_papers"


print(f"Data directory exists: {data_dir.exists()}")
print(f"Sample docs directory: {sample_docs_dir.exists()}")
print(f"Research papers directory: {research_papers_dir.exists()}")

if research_papers_dir.exists():
    files = list(research_papers_dir.glob("*.pdf"))
    print(f"\nFound {len(files)} PDF files in research_papers/")
    for f in files:
        print(f"  - {f.name}")

Data directory exists: True
Sample docs directory: True
Research papers directory: True

Found 1 PDF files in research_papers/
  - 2408.09869v5.pdf


### SimpleDirectoryReader Features

**Key Parameters:**
- `input_dir`: Directory path
- `required_exts`: Filter by extensions (e.g., `[".pdf", ".txt"]`)
- `recursive`: Scan subdirectories
- `filename_as_id`: Use filename as document ID
- `file_metadata`: Custom metadata function
- `exclude_hidden`: Skip hidden files

In [5]:
# Load real PDFs from data/research_papers/ when present; otherwise create
# synthetic sample papers. Downstream cells all operate on `sample_papers`.
pdf_dir = Path("./data/research_papers")

if any(pdf_dir.glob("*.pdf")):
    reader = SimpleDirectoryReader(
        input_dir=str(pdf_dir),
        required_exts=[".pdf"],          # only PDFs
        recursive=True,                   # include subdirectories
        filename_as_id=True,              # use the filename as the document id
    )
    sample_papers = reader.load_data()
    print(f"✅ Loaded {len(sample_papers)} PDF chunk(s) from data/research_papers/")
    for doc in sample_papers[:5]:
        print(f"  - {doc.metadata.get('file_name', doc.doc_id)} ({len(doc.text)} chars)")
    if len(sample_papers) > 5:
        print(f"  ... and {len(sample_papers) - 5} more")
else:
    print("ℹ️  No PDFs found in data/research_papers/ — creating synthetic sample papers.")
    print("   Drop real PDFs into that folder and re-run this cell to load them instead.")
    sample_papers = [
        Document(
            text="""
            Title: Attention Is All You Need
            Authors: Vaswani et al.
            Year: 2017

            Abstract: The dominant sequence transduction models are based on complex recurrent or
            convolutional neural networks that include an encoder and a decoder. The best performing
            models also connect the encoder and decoder through an attention mechanism. We propose a
            new simple network architecture, the Transformer, based solely on attention mechanisms,
            dispensing with recurrence and convolutions entirely.

            Introduction: Recurrent neural networks, long short-term memory and gated recurrent neural
            networks in particular, have been firmly established as state of the art approaches in
            sequence modeling and transduction problems. The Transformer is the first transduction model
            relying entirely on self-attention to compute representations of its input and output without
            using sequence-aligned RNNs or convolution.
            """,
            metadata={
                "title": "Attention Is All You Need",
                "authors": "Vaswani et al.",
                "year": 2017,
                "category": "transformers",
                "citations": 85000,
                "source": "research_paper"
            }
        ),
        Document(
            text="""
            Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
            Authors: Devlin et al.
            Year: 2019

            Abstract: We introduce a new language representation model called BERT, which stands for
            Bidirectional Encoder Representations from Transformers. Unlike recent language representation
            models, BERT is designed to pre-train deep bidirectional representations from unlabeled text
            by jointly conditioning on both left and right context in all layers.

            Introduction: Language model pre-training has been shown to be effective for improving many
            natural language processing tasks. Pre-trained language representations can be either context-free
            or context-based. BERT alleviates the unidirectionality constraint using a masked language
            model (MLM) pre-training objective.
            """,
            metadata={
                "title": "BERT",
                "authors": "Devlin et al.",
                "year": 2019,
                "category": "language_models",
                "citations": 65000,
                "source": "research_paper"
            }
        ),
        Document(
            text="""
            Title: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
            Authors: Lewis et al.
            Year: 2020

            Abstract: Large pre-trained language models have been shown to store factual knowledge in their
            parameters, and achieve state-of-the-art results when fine-tuned on downstream NLP tasks. However,
            their ability to access and precisely manipulate knowledge is still limited. We explore a general
            fine-tuning recipe for retrieval-augmented generation (RAG) models which combine parametric and
            non-parametric memory.

            Introduction: Pre-trained neural language models store and retrieve knowledge using their parameters.
            RAG models combine parametric memory (the LLM) with non-parametric memory (a dense vector index of
            Wikipedia). This provides the model with access to up-to-date information and allows for more
            interpretable and modular systems.
            """,
            metadata={
                "title": "RAG",
                "authors": "Lewis et al.",
                "year": 2020,
                "category": "rag",
                "citations": 3500,
                "source": "research_paper"
            }
        ),
    ]
    print(f"✅ Created {len(sample_papers)} sample research papers")
    for doc in sample_papers:
        print(f"  - {doc.metadata['title']} ({doc.metadata['year']})")

✅ Loaded 9 PDF chunk(s) from data/research_papers/
  - 2408.09869v5.pdf (2193 chars)
  - 2408.09869v5.pdf (3505 chars)
  - 2408.09869v5.pdf (3122 chars)
  - 2408.09869v5.pdf (4113 chars)
  - 2408.09869v5.pdf (3655 chars)
  ... and 4 more


### 2.2 Custom Metadata Functions

In [7]:
# Add processing metadata
for doc in sample_papers:
    doc.metadata["processed_date"] = datetime.now().isoformat()
    doc.metadata["char_count"] = len(doc.text)
    doc.metadata["word_count"] = len(doc.text.split())

print("Enhanced metadata for first document:")
for key, value in sample_papers[0].metadata.items():
    print(f"  {key}: {value}")

Enhanced metadata for first document:
  page_label: 1
  file_name: 2408.09869v5.pdf
  file_path: f:\llama-index-tutorials\data\research_papers\2408.09869v5.pdf
  file_type: application/pdf
  file_size: 5566575
  creation_date: 2026-11-21
  last_modified_date: 2026-11-21
  processed_date: 2026-11-21T18:56:02.233974
  char_count: 2193
  word_count: 312


---

## 3. Chunking Strategies

### Why Chunking Matters

Chunking is **critical** for RAG quality:

1. **Context Window Limits**: LLMs have token limits
2. **Embedding Quality**: Smaller chunks = more focused embeddings
3. **Retrieval Precision**: Granular chunks improve relevance
4. **Cost Optimization**: Smaller chunks = fewer tokens to LLM

### 3.1 Sentence-Based Chunking

In [9]:
# SentenceSplitter: Respects sentence boundaries
sentence_splitter = SentenceSplitter(
    chunk_size=1024,     # Target tokens per chunk
    chunk_overlap=200,   # Overlap to preserve context
    separator=" ",       # Split on spaces first
)


sentence_nodes = sentence_splitter.get_nodes_from_documents(sample_papers)

print(f"SentenceSplitter Results:")
print(f"  Input documents: {len(sample_papers)}")
print(f"  Output nodes: {len(sentence_nodes)}")
print(f"  Avg chars per node: {sum(len(n.text) for n in sentence_nodes) / len(sentence_nodes):.0f}")

print(f"\nFirst node preview:")
print(f"  Text (first 200 chars): {sentence_nodes[0].text[:200]}...")
print(f"  Metadata: {sentence_nodes[0].metadata}")

SentenceSplitter Results:
  Input documents: 9
  Output nodes: 19
  Avg chars per node: 2871

First node preview:
  Text (first 200 chars): Docling Technical Report
Version 1.0
Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos
Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer
Kasper Dinkla Lokes...
  Metadata: {'page_label': '1', 'file_name': '2408.09869v5.pdf', 'file_path': 'f:\\llama-index-tutorials\\data\\research_papers\\2408.09869v5.pdf', 'file_type': 'application/pdf', 'file_size': 5566575, 'creation_date': '2026-11-21', 'last_modified_date': '2026-11-21', 'processed_date': '2026-11-21T18:56:02.233974', 'char_count': 2193, 'word_count': 312}


### 🎯 ML Engineering Note: Chunk Size Selection

**Chunk Size Trade-offs:**

| Size | Pros | Cons | Use Case |
|------|------|------|----------|
| **Small (256-512)** | Precise retrieval, lower cost | May lose context | Q&A, factoid extraction |
| **Medium (512-1024)** | Balanced context/precision | Good default | General RAG, document QA |
| **Large (1024-2048)** | Rich context | Diluted relevance, higher cost | Summarization, broad queries |

**Overlap Guidelines:**
- 10-20% of chunk size (typical)
- Higher overlap (20-30%) for dense, technical content
- Lower overlap (5-10%) for structured documents